In [ ]:
# Colab setup: fetch the data and the helper file (skip if you run locally with the files here)
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture6/main/osterild_fast_uvw.csv -O osterild_fast_uvw.csv
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture6/main/l6_helpers.py -O l6_helpers.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from l6_helpers import *

case = "moderate"                       # "strong" in Bonus A
t, u = load_block(case, block=0)        # one 10-min block of u'(t), mean removed
U = U_MEAN[case]                        # mean wind speed (Lecture 5)
N = u.size                              # number of samples
T_rec = N * DT                          # record length T (s)
dw = 2 * np.pi / T_rec                  # frequency spacing (rad/s)
print(f"{case}: U = {U} m/s, N = {N} samples, T = {T_rec:.0f} s, var(u) = {np.var(u):.3f} m²/s²")

# Round 1 — The Fourier transform, on real wind
### 46100 Micrometeorology · Lecture 6

In the game, the analyser multiplied the signal by one frequency and added up over the record. In the notes that is
eq. 4.3.10, and with samples every $\Delta t$, eq. 4.3.14:

$$X(\omega_l,T) = \frac{1}{2\pi}\int_0^T x(t)\,e^{-i\omega_l t}\,dt \;\approx\; \frac{1}{2\pi}\sum_{n} x(n\Delta t)\,e^{-i\omega_l n\Delta t}\,\Delta t, \qquad \omega_l = \frac{2\pi l}{T}.$$

You compute it for one frequency, find out how `np.fft` does it for all frequencies at once, check Parseval,
and turn frequencies into eddy sizes, on ten minutes of real Østerild wind.

**Time: ~27' work + 3' pair answers.** Work in pairs (File → *Save a copy in Drive*, share it with your partner).

| Task | Time | What you'll do |
|---|---|---|
| 1 | 3' | Look at the wind and commit two guesses |
| 2 | 7' | $X(\omega_l,T)$ for one frequency |
| 3 | 7' | The FFT: all frequencies at once |
| 4 | 6' | Parseval: does $\sum S\,\Delta\omega$ give the variance? |
| 5 | 4' | From frequencies to eddy sizes in metres |

> **Getting stuck is part of this.** These tasks are designed so that you work it out; nobody expects you to
> know the answer beforehand. Stuck for 3 minutes? Open hint 1. **Call the teacher or TA** when you've opened two hints,
> or a check FAILS and you can't see why. Tell us what you expected and what you got.

## Task 1 — Look, then commit (3')
Before computing anything, commit two guesses in the cell below.
- Counting frequencies from the lowest up (l = 1, 2, 3, …), **how many do you need for 90 % of the variance?** (1 to 6000)
- In Lecture 5 you measured $L_u$ ≈ 136 m for this hour. **Which eddy size (m) do you expect to hold the most variance?**

In [ ]:
plt.figure(figsize=(11, 3)); plt.plot(t, u, lw=0.5)
plt.xlabel("t (s)"); plt.ylabel("u′ (m/s)"); plt.title(f"Østerild 103 m, {case} wind, one 10-min block"); plt.show()

my_guess_n90 = None        # number of frequencies for 90 % of the variance
my_guess_size_m = None     # eddy size (m) holding the most variance

## Task 2 — $X(\omega_l,T)$ for one frequency (7')
`wave` below is $e^{i\omega_l t}$, sampled at the same times as the wind.
1. Write `X_l`: eq. 4.3.14 for this one frequency (one line).
2. Run. The record rebuilt from this frequency and its mirror $-\omega_l$ is $2\,\mathrm{Re}\big(X(\omega_l,T)\,e^{i\omega_l t}\big)\,\Delta\omega$
   (the "rebuild" row of the slides); it is plotted on top of the wind.
3. Try several `l` (1, 3, 10, 100). Which removes the most variance?

In [ ]:
l = 3
w_l = 2 * np.pi * l / T_rec                 # omega_l (rad/s)
wave = np.exp(1j * w_l * t)                 # e^{i omega_l t}, sampled

# your code here: eq. 4.3.14 for this one frequency
X_l = None
assert X_l is not None, "Write X_l first."

piece = 2 * np.real(X_l * wave) * dw        # this frequency and its mirror -omega_l
print(f"l = {l}: period {T_rec / l:.1f} s, |X(omega_l,T)| = {abs(X_l):.3f} m")
print(f"variance removed by this one piece: {1 - np.var(u - piece) / np.var(u):.1%}")
other = np.exp(1j * 2 * np.pi * (l + 1) * t / T_rec)
print(f"(check) (1/T) * integral of e^(i w_l t) e^(-i w_(l+1) t) dt = {abs(np.mean(wave * np.conj(other))):.1e}  <- orthogonal")
plt.figure(figsize=(11, 3)); plt.plot(t, u, lw=0.5, label="u′"); plt.plot(t, piece, lw=2, label=f"l = {l}")
plt.xlabel("t (s)"); plt.legend(); plt.show()

<details><summary><b>Hint 1</b></summary>

Eq. 4.3.14: multiply the signal by $e^{-i\omega_l t}$, sum over the samples, multiply by $\Delta t/2\pi$.

</details>
<details><summary><b>Hint 2</b></summary>

$e^{-i\omega_l t}$ is `np.conj(wave)`; $\Delta t$ is `DT`; the sum is `np.sum(...)`.

</details>
<details><summary><b>Hint 3</b></summary>

`X_l = np.sum(u * np.conj(wave)) * DT / (2*np.pi)`

</details>

## Task 3 — The FFT: all frequencies at once (7')
`np.fft.fft` does the same multiply-and-add for every $l$ in one go, but without the constants in front.
The notes call its output $X_{FFT}$ (eq. 4.3.17).
1. Write `X_fft`, the FFT of `u`. The cell compares `X_fft[l]` with your `X_l` from Task 2.
2. From the ratio, work out the constants and write `X`: $X(\omega_l,T)$ for *all* $l$. The check must PASS.

In [ ]:
# your code here: the FFT of the whole record
X_fft = None
assert X_fft is not None, "Write X_fft first."
print(f"l = {l}: FFT gives {X_fft[l]:.3f}, your X_l = {X_l:.5f}, ratio FFT / yours = {X_fft[l] / X_l:.6g}")

# your code here: X(omega_l, T) for all l, from X_fft
X = None
assert X is not None, "Write X first."
ok = np.isclose(X[l], X_l)
print(f"CHECK X[{l}] = {X[l]:.5f} vs your X_l = {X_l:.5f} -> {'PASS' if ok else 'FAIL'}")

<details><summary><b>Hint 1</b></summary>

Look at the ratio it printed. Which constants did you put in front of the sum in Task 2?

</details>
<details><summary><b>Hint 2</b></summary>

Compare the ratio with $2\pi/\Delta t$ (= `2*np.pi/DT`). `np.fft.fft` returns the bare sum $\sum_n x\,e^{-i2\pi ln/N}$.

</details>
<details><summary><b>Hint 3</b></summary>

`X = X_fft * DT / (2*np.pi)`

</details>

*So the FFT is eq. 4.3.14 for all N frequencies at once, without $\Delta t/2\pi$: different software, different constants
(the notes warn about this after eq. 4.3.16). Always check which ones your tool uses.*

## Task 4 — Parseval: does $\sum S\,\Delta\omega$ give the variance? (6')
On the slides, Parseval gave $\sigma^2 = \sum_l S(\omega_l,T)\,\Delta\omega$ with $S(\omega_l,T) = \frac{2\pi}{T}X X^*$ (eq. 4.3.9).
1. **Commit first:** will the sum be *smaller than*, *equal to* or *larger than* `np.var(u)`?
2. Write `S_T` (eq. 4.3.9, for all $l$) and the sum. Then run: the plot also answers your Task 1 guess.

In [ ]:
my_guess_sum = None                   # "smaller", "equal" or "larger"

# your code here: S(omega_l, T) for all l (eq. 4.3.9), and the sum of S * d_omega over all l
S_T = None
total = None
assert None not in (S_T, total, my_guess_sum), "Commit a guess and write S_T and total first."
print(f"your guess: {my_guess_sum}\nsum of S * d_omega = {total:.4f} m²/s²\nvariance np.var(u) = {np.var(u):.4f} m²/s²")

# each positive frequency and its mirror together carry 2 S d_omega
share = 2 * S_T[1:N // 2] * dw / np.var(u)
frac = np.cumsum(share)
n90 = np.argmax(frac >= 0.9) + 1
compare("frequencies for 90 % of the variance", my_guess_n90, n90)
plt.semilogx(np.arange(1, N // 2), frac); plt.axhline(0.9, c="k", ls="--")
plt.xlabel("frequencies kept (lowest first)"); plt.ylabel("share of var(u)"); plt.show()

<details><summary><b>Hint 1</b></summary>

$X X^* = |X|^2$: `np.abs(X)**2`. $T$ is `T_rec`, $\Delta\omega$ is `dw`.

</details>
<details><summary><b>Hint 2</b></summary>

Sum over **all** N entries: the negative frequencies are in the second half of the array (two-sided spectrum).

</details>
<details><summary><b>Hint 3</b></summary>

`S_T = 2*np.pi/T_rec * np.abs(X)**2` and `total = np.sum(S_T) * dw`

</details>

*Why can't two different frequencies add cross terms? (The orthogonality check in Task 2, and Pythagoras.)* …

## Task 5 — From frequencies to eddy sizes (4')
Frequency $l$ repeats every $T/l$ seconds. Write the eddy size $\lambda$ it stands for (Taylor: size = wind × period),
then compare the tallest bar with your guess and with $L_u$.

In [ ]:
l_pos = np.arange(1, N // 2)                       # positive frequencies
var_modes = 2 * S_T[1:N // 2] * dw                 # variance of each frequency (with its mirror)

# your code here: the eddy size (m) of every positive frequency
lam_l = None
assert lam_l is not None, "Write lam_l first."

centres, band_var, edges = size_bands(lam_l, var_modes)
peak = centres[np.argmax(band_var)]
compare("eddy size holding the most variance (m)", my_guess_size_m, peak)
print(f"L_u (Lecture 5) = {L_U_LECTURE5[case]:.0f} m, ratio peak / L_u = {peak / L_U_LECTURE5[case]:.1f}")
plt.figure(figsize=(11, 3.5)); plt.bar(centres, band_var / np.var(u), width=np.diff(edges) * 0.85, color="#2F3EEA")
plt.xscale("log"); plt.gca().invert_xaxis(); plt.axvline(L_U_LECTURE5[case], c="#990000", ls="--", label="L_u")
plt.xlabel("eddy size λ (m)  ← big eddies left"); plt.ylabel("share of var(u)"); plt.legend(); plt.show()

<details><summary><b>Hint 1</b></summary>

Period of frequency $l$: $T/l$. An eddy that takes that long to pass at speed U has size …?

</details>
<details><summary><b>Hint 2</b></summary>

size = U × period.

</details>
<details><summary><b>Hint 3</b></summary>

`lam_l = U * T_rec / l_pos`

</details>

---
### Your pair's answers (3') — for the report-out
1. How many frequencies for 90 % of the variance, and how far off was your guess?
2. What did $\sum S\,\Delta\omega$ give? Why is that useful?
3. Which eddy size holds the most variance? Is it $L_u$?

---
## Bonus (only if you finish early)
**A.** Set `case = "strong"` in the setup cell and rerun. Does the tallest bar move to bigger eddies, to higher frequencies, or both?
**B.** Use `block=3`. How much do the bars change between two blocks of the same hour?